# Day 06 — Faithfulness, Hallucination & Answer Relevancy

**Module 2 · The Metric Toolkit**

Today we explore three important metrics for evaluating generated answers:

- **Faithfulness** — Is the answer supported by the provided context?
- **Hallucination** — Does the answer contain unsupported or contradictory information?
- **Answer Relevancy** — Does the answer actually address the user's question?

We will use the same context and evaluate different answers against it.

> **Core idea:** Different metrics answer different evaluation questions.

## 1. Setup

We will use the same judge model from the previous days.

In [1]:
import os

from dotenv import load_dotenv
from deepeval.models import LocalModel

load_dotenv()

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found."

judge = LocalModel(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

print("Judge:", judge.get_model_name())

Judge: openai/gpt-oss-120b (Local Model)


## 2. Create Our Context

Imagine that our AI application retrieved these two pieces of information from a company's knowledge base.

This information will be used to evaluate the generated answers.

In [2]:
from deepeval.test_case import LLMTestCase

context = [
    "The refund window is 30 days from the purchase date.",
    "Digital products are refundable only if never downloaded.",
]

test_cases = [
    LLMTestCase(
        input="Can I get a refund on an ebook I never downloaded?",
        actual_output=(
            "Yes. Refunds are available within 30 days, and digital products "
            "are refundable if they were never downloaded."
        ),
        context=context,
        retrieval_context=context,
    ),
    LLMTestCase(
        input="Can I get a refund on an ebook I never downloaded?",
        actual_output=(
            "Yes. You can get a refund within 30 days. "
            "We will also give you a free physical copy and a lifetime subscription."
        ),
        context=context,
        retrieval_context=context,
    ),
    LLMTestCase(
        input="Can I get a refund on an ebook I never downloaded?",
        actual_output=(
            "The weather in Portugal is lovely this time of year. "
            "Our office has four floors and a great espresso machine."
        ),
        context=context,
        retrieval_context=context,
    ),
]

print(f"Created {len(test_cases)} test cases.")

Created 3 test cases.


## 3. Faithfulness

**Question:**

> Is the generated answer supported by the provided context?

Faithfulness is especially important for applications such as RAG, where the answer is expected to stay grounded in retrieved information.

```text
Retrieval Context
       ↓
Generated Answer
       ↓
Is the answer supported?

In [3]:
from deepeval.metrics import FaithfulnessMetric

faithfulness = FaithfulnessMetric(
    model=judge,
)

print("Faithfulness metric created.")

Faithfulness metric created.


## 4. Hallucination

**Question:**

> Does the answer contain information that is not supported by the context or conflicts with it?

Hallucination evaluation helps us identify when an AI system introduces information that it should not have generated.

```text
Context
   ↓
Answer
   ↓
Unsupported / conflicting information?

In [4]:
from deepeval.metrics import HallucinationMetric

hallucination = HallucinationMetric(
    model=judge,
)

print("Hallucination metric created.")

Hallucination metric created.


C:\Users\T14s\AppData\Local\Temp\ipykernel_6212\1025934349.py:3: DeprecationWarning: 'HallucinationMetric' now scores in the same direction as every other deepeval metric: 1 is a pass, 0 is a failure, and 'threshold' is the MINIMUM passing score. It previously scored the proportion of violations, where 'threshold' was a maximum. Review any 'threshold' you pass and any code reading '.score' - a threshold of 0.2 that used to mean 'at most 20% violations' should now be 0.8. This notice will be removed in a future release.
  hallucination = HallucinationMetric(


## 5. Answer Relevancy

**Question:**

> Does the generated answer actually address the user's question?

Unlike the previous two metrics, Answer Relevancy does not require retrieval context.

It focuses on the relationship between:

```text
User Question
      ↓
Generated Answer
      ↓
Is the answer relevant?

In [5]:
from deepeval.metrics import AnswerRelevancyMetric

answer_relevancy = AnswerRelevancyMetric(
    model=judge,
)

print("Answer Relevancy metric created.")

Answer Relevancy metric created.


## 6. Run All Three Metrics

Now we evaluate the same test cases using all three metrics.

In [6]:
from deepeval import evaluate

metrics = [
    faithfulness,
    hallucination,
    answer_relevancy,
]

results = evaluate(
    test_cases=test_cases,
    metrics=metrics,
)

for i, result in enumerate(results.test_results, start=1):
    print(f"\n{'=' * 60}")
    print(f"Test Case {i}")

    for metric_result in result.metrics_data:
        print(
            f"{metric_result.name:<20} "
            f"score={metric_result.score:.2f} "
            f"success={metric_result.success}"
        )
        print(f"Reason: {metric_result.reason}")

✨ You're running DeepEval's latest Faithfulness Metric! (using openai/gpt-oss-120b (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Hallucination Metric! (using openai/gpt-oss-120b (Local Model), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using openai/gpt-oss-120b (Local Model), 
strict=False, async_mode=True)...

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

RetryError: RetryError[<Future at 0x1df88211810 state=finished raised RateLimitError>]

## 7. How Are These Metrics Different?

The three metrics look at different dimensions of the same answer.

| Metric | Main Question |
|---|---|
| Faithfulness | Is the answer supported by the context? |
| Hallucination | Did the answer introduce unsupported or conflicting information? |
| Answer Relevancy | Does the answer address the user's question? |

This distinction is important.

An answer can be:

- relevant but not grounded,
- grounded but irrelevant,
- or both relevant and grounded.

Therefore, **one metric is rarely enough to describe the quality of an AI application.**

# Day 06 — Key Takeaways

Today we learned three important evaluation dimensions:

### Faithfulness
Checks whether the answer is supported by the provided context.

### Hallucination
Checks for unsupported or conflicting information in the answer.

### Answer Relevancy
Checks whether the answer addresses the user's question.

The important mental model is:

```text
                 AI Answer
                     │
        ┌────────────┼────────────┐
        ↓            ↓            ↓
   Faithfulness  Hallucination  Relevancy
        │            │            │
   Grounded?     Invented?     Useful?